# ПЗ-4. Разрешение и качество на документах

## 1. Цель
Измерить ANLS, Exact Match и время ответа при трёх ограничениях разрешения.

## 2. Датасеты и модели
100 вопросов DocVQA validation из [lmms-lab-encoder/DocVQA](https://huggingface.co/datasets/lmms-lab-encoder/DocVQA). Старое имя lmms-lab/DocVQA перенаправляется сюда. Выборка фиксируется до получения ответов. Публичность набора не гарантирует отсутствия персональных данных: ниже обязателен просмотр страниц. В модель передаются только одобренные документы без таких данных.

[Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct), fp16. [Документация процессора](https://huggingface.co/docs/transformers/v4.57.1/model_doc/qwen2_5_vl).

Источник задания: `Практика_06_11_2026_1_week.docx`. Приложения А.1–А.7 отсутствуют; ниже приведены стандартные реализации. LoRA-проект не входит в эти шесть работ.

## 3. Подготовка окружения
Откройте этот notebook в Google Colab. Для ПЗ-1 выберите CPU, для ПЗ-3–6 — T4; для ПЗ-2 GPU ускорит получение признаков. Выполните установку, перезапустите сеанс, затем выполните остальные ячейки сверху вниз. Все результаты сохраняются в папке задания; скачайте её вместе с ноутбуком. Данные и веса кэшируются в соседней папке `data`. При переносе в Colab прежние результаты доступны только после загрузки файлов. GPU использует fp16. Числа в сохранённых выводах относятся только к реально выполненным ячейкам. Ячейки без execution_count не запускались.

In [ ]:
%pip install torch==2.9.1 torchvision==0.24.1 transformers==4.57.6 numpy==2.3.5 pandas==2.3.3 matplotlib==3.10.8 pillow==12.0.0 datasets==4.4.1 accelerate==1.12.0 sentencepiece==0.2.1 scikit-learn==1.7.2 pycocoevalcap==1.2 reportlab==4.4.5 huggingface-hub==0.36.2 safetensors==0.8.0 tokenizers==0.22.2 protobuf==6.33.2 ftfy==6.3.1 requests==2.34.2 fsspec==2025.10.0 pyarrow==26.0.1

In [1]:
LAB = 'PZ_4'
REVISIONS = {'facebook/dino-vitb16': 'f205d5d8e640a89a2b8ef0369670dfc37cc07fc2', 'openai/clip-vit-base-patch32': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268', 'google/siglip-base-patch16-224': '7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed', 'Qwen/Qwen2.5-0.5B-Instruct': '7ae557604adf67be50417f59c2c2f167def9a775', 'Qwen/Qwen2.5-VL-3B-Instruct': '66285546d2b821cf421d4f5eb2576359d3770cd3', 'lmms-lab-encoder/DocVQA': '539088ef8a8ada01ac8e2e6d4e372586748a265e', 'lmms-lab-encoder/RefCOCO': 'a5dff0b3194715fda69b2d6a0d2aaafb41eaa407'}

import os
import sys
import json
import random
import time
import math
import gc
import hashlib
import platform
import importlib.metadata
from pathlib import Path
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor

ROOT = Path.cwd() if Path.cwd().name == LAB else Path.cwd() / LAB
ROOT.mkdir(exist_ok=True)
OUT = ROOT / "results"
OUT.mkdir(exist_ok=True)
DATA = ROOT.parent / "data"
DATA.mkdir(exist_ok=True)
os.environ["HF_HOME"] = str(DATA / "huggingface")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(8, os.cpu_count() or 1))
torch.use_deterministic_algorithms(True, warn_only=True)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32

def save_json(name, value):
    path = OUT / name
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    for attempt in range(20):
        try:
            temporary.replace(path)
            return
        except PermissionError:
            if attempt==19:
                raise
            time.sleep(0.1)

def save_figure(name):
    plt.savefig(OUT / name, dpi=130, bbox_inches="tight")
    plt.show()

versions = {p.metadata["Name"]: p.version for p in importlib.metadata.distributions() if p.metadata["Name"]}
environment = {"python": sys.version, "platform": platform.platform(), "seed": SEED,
               "device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
               "versions": versions, "revisions": REVISIONS}
save_json("environment.json", environment)
print({k: v for k, v in environment.items() if k != "versions"})

C:\Users\darri\PycharmProjects\cv061126\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.14.5 (tags/v3.14.5:5607950, May 10 2026, 10:43:50) [MSC v.1944 64 bit (AMD64)]', 'platform': 'Windows-11-10.0.26200-SP0', 'seed': 42, 'device': 'cpu', 'gpu': None, 'revisions': {'facebook/dino-vitb16': 'f205d5d8e640a89a2b8ef0369670dfc37cc07fc2', 'openai/clip-vit-base-patch32': '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268', 'google/siglip-base-patch16-224': '7fd15f0689c79d79e38b1c2e2e2370a7bf2761ed', 'Qwen/Qwen2.5-0.5B-Instruct': '7ae557604adf67be50417f59c2c2f167def9a775', 'Qwen/Qwen2.5-VL-3B-Instruct': '66285546d2b821cf421d4f5eb2576359d3770cd3', 'lmms-lab-encoder/DocVQA': '539088ef8a8ada01ac8e2e6d4e372586748a265e', 'lmms-lab-encoder/RefCOCO': 'a5dff0b3194715fda69b2d6a0d2aaafb41eaa407'}}


In [2]:
from xml.sax.saxutils import escape
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as PDFImage
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib import colors
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from matplotlib import font_manager

def write_report(title, goal, work, findings, table=None, figure=None):
    pdfmetrics.registerFont(TTFont("Student", font_manager.findfont("DejaVu Sans")))
    style = ParagraphStyle("Student", fontName="Student", fontSize=10, leading=14, spaceAfter=8)
    heading = ParagraphStyle("Heading", parent=style, fontSize=14, leading=18, spaceAfter=12)
    story = [Paragraph(escape(title), heading), Paragraph(escape("Цель. " + goal), style),
             Paragraph(escape("Что было сделано. " + work), style)]
    for text in findings:
        story.append(Paragraph(escape(text), style))
    if table is not None and len(table):
        rows = [[Paragraph(escape(str(x)), style) for x in table.columns]]
        rows += [[Paragraph(escape(f"{x:.4f}" if isinstance(x, float) else str(x)), style) for x in row]
                 for row in table.itertuples(index=False, name=None)]
        item = Table(rows, repeatRows=1, colWidths=[495 / len(table.columns)] * len(table.columns))
        item.setStyle(TableStyle([("VALIGN", (0,0), (-1,-1), "TOP"),
                                 ("GRID", (0,0), (-1,-1), 0.3, colors.grey)]))
        story.extend([item, Spacer(1, 10)])
    if figure is not None and Path(figure).exists():
        with Image.open(figure) as im:
            ratio = im.height / im.width
        story.append(PDFImage(str(figure), width=460, height=min(250, 460 * ratio)))
    story.append(Paragraph(escape("Индивидуальный вклад. Подготовлены реализации, протоколы экспериментов, тесты и оформление. Код и текст подготовлены с помощью ИИ-ассистента; фактические запуски и ограничения перечислены выше. Авторство невыполненных опытов не заявляется."), style))
    story.append(Paragraph(escape("Воспроизведение: notebook " + LAB + ".ipynb; seed=42. Версии среды и точные ревизии весов сохранены в results/environment.json. Контрольные вопросы разобраны в ноутбуке."), style))
    SimpleDocTemplate(str(ROOT / "report.pdf"), pagesize=(595,842), rightMargin=50, leftMargin=50,
                      topMargin=45, bottomMargin=45).build(story)
    print("PDF:", ROOT / "report.pdf")

## 4. Реализация
ANLS = max по эталонам от 1−d/max(len), если нормированное расстояние строго меньше 0.5; иначе 0. Используются lower и strip; числовые строки не преобразуются в float. Поэтому замена одной цифры может сохранить высокий ANLS и дать EM=0.

In [3]:
import re

def normalize_answer(text):
    return str(text).strip().lower()

def edit_distance(a,b):
    previous = list(range(len(b)+1))
    for i, char_a in enumerate(a,1):
        current = [i]
        for j,char_b in enumerate(b,1):
            current.append(min(current[-1]+1,previous[j]+1,previous[j-1]+(char_a!=char_b)))
        previous = current
    return previous[-1]

def anls(prediction,references):
    prediction = normalize_answer(prediction)
    values = []
    for reference in references:
        reference = normalize_answer(reference)
        maximum = max(len(prediction),len(reference))
        distance = edit_distance(prediction,reference)/maximum if maximum else 0.
        values.append(1.-distance if distance < 0.5 else 0.)
    return max(values,default=0.)

def exact_match(prediction,references):
    return float(any(normalize_answer(prediction)==normalize_answer(reference) for reference in references))

def numeric_answer(references):
    return bool(references) and all(bool(re.search(r"\d",str(s))) and bool(re.fullmatch(r"[+\-−\d\s.,:/%$€£¥()]+",str(s).strip())) for s in references)

PIXELS = {"0.25 MP":int(250000//784*784),"1 MP":int(1000000//784*784),"4 MP":int(4000000//784*784)}

In [4]:
from transformers.models.qwen2_vl.image_processing_qwen2_vl import smart_resize
assert anls("1234",["1235"]) == 0.75
assert exact_match("1234",["1235"]) == 0.
assert anls("ab",["ac"]) == 0.
assert anls("",[""]) == 1.
assert anls("",["abc"]) == 0.
assert anls(" answer ",["ANSWER","wrong"]) == 1.
assert not numeric_answer([])
assert numeric_answer(["$1,250.50"]) and not numeric_answer(["room 5"])
resize_tests = []
for cap in PIXELS.values():
    for h,w in [(1000,750),(3000,2000),(337,2001)]:
        rh,rw = smart_resize(h,w,factor=28,min_pixels=4*784,max_pixels=cap)
        assert rh%28 == 0 and rw%28 == 0 and rh*rw <= cap
        resize_tests.append({"height":h,"width":w,"cap":cap,"resized_height":rh,"resized_width":rw,"tokens":rh*rw//784})
display(pd.DataFrame(resize_tests))
save_json("unit_tests.json",{"ANLS_1234_1235":anls("1234",["1235"]),"EM_1234_1235":0.,"resize_cases":resize_tests,"passed":True})

,height,width,cap,resized_height,resized_width,tokens
0,1000,750,249312,560,420,300
1,3000,2000,249312,588,392,294
2,337,2001,249312,196,1204,301
3,1000,750,999600,1008,756,972
4,3000,2000,999600,1204,812,1247
5,337,2001,999600,336,1988,852
6,1000,750,3999968,1008,756,972
7,3000,2000,3999968,2436,1624,5046
8,337,2001,3999968,336,1988,852


In [5]:
from transformers import AutoProcessor
model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
processor_test = AutoProcessor.from_pretrained(model_id,revision=REVISIONS[model_id],use_fast=False,
                                              min_pixels=4*784,max_pixels=PIXELS["0.25 MP"])
synthetic = Image.new("RGB",(1500,2000),"white")
messages_test = [{"role":"user","content":[{"type":"image"},{"type":"text","text":"Read the image."}]}]
chat_test = processor_test.apply_chat_template(messages_test,tokenize=False,add_generation_prompt=True)
processor_rows = []
for name,cap in PIXELS.items():
    processor_test.image_processor.max_pixels = cap
    batch_test = processor_test(text=[chat_test],images=[synthetic],return_tensors="pt",
                                images_kwargs={"min_pixels":4*784,"max_pixels":cap})
    grid = batch_test.image_grid_thw[0].tolist()
    tokens = int(np.prod(grid)//processor_test.image_processor.merge_size**2)
    token_id = processor_test.tokenizer.convert_tokens_to_ids("<|image_pad|>")
    assert int((batch_test.input_ids==token_id).sum())==tokens
    assert grid[1]*14%28==0 and grid[2]*14%28==0 and grid[1]*grid[2]*196<=cap
    expected_size = smart_resize(2000,1500,factor=28,min_pixels=4*784,max_pixels=cap)
    assert (grid[1]*14,grid[2]*14)==expected_size
    processor_rows.append({"resolution":name,"max_pixels":cap,"height":grid[1]*14,"width":grid[2]*14,"visual_tokens":tokens})
display(pd.DataFrame(processor_rows))
save_json("processor_tests.json",processor_rows)
del processor_test,batch_test

,resolution,max_pixels,height,width,visual_tokens
0,0.25 MP,249312,560,420,300
1,1 MP,999600,1148,840,1230
2,4 MP,3999968,1988,1512,3834


In [5]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

def load_vlm(max_pixels=1000000):
    if DEVICE.type != "cuda":
        raise RuntimeError("Для полного эксперимента требуется GPU T4 16 GB или совместимая CUDA GPU. Размер выборки не уменьшен.")
    model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
    processor = AutoProcessor.from_pretrained(model_id, revision=REVISIONS[model_id],
                                             min_pixels=4*28*28, max_pixels=max_pixels, use_fast=False)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id, revision=REVISIONS[model_id], torch_dtype=torch.float16,
        attn_implementation="sdpa").to(DEVICE).eval()
    model.requires_grad_(False)
    return model, processor

def vlm_answer(model, processor, image, prompt, max_new_tokens=64):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    chat = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    batch = processor(text=[chat], images=[image], return_tensors="pt",
                      images_kwargs={"min_pixels":processor.image_processor.min_pixels,
                                     "max_pixels":processor.image_processor.max_pixels})
    grid = batch["image_grid_thw"][0].tolist()
    n_visual = int(np.prod(grid) // processor.image_processor.merge_size**2)
    batch = batch.to(DEVICE)
    batch["pixel_values"] = batch["pixel_values"].to(torch.float16)
    torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**batch, do_sample=False, max_new_tokens=max_new_tokens,
                                   use_cache=True)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    text = processor.batch_decode(generated[:, batch.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    return {"answer": text, "visual_tokens": n_visual, "seconds": elapsed,
            "grid_thw": grid, "processed_height": grid[1]*14, "processed_width": grid[2]*14,
            "generated_tokens": generated.shape[1]-batch.input_ids.shape[1]}

## 5. Эксперименты
Сначала просмотр и фиксация ровно 100 вопросов. Одобрение относится к полной странице, вопросу и эталонам. При наличии имён частных лиц, подписей, личных контактов и других персональных сведений страницу нужно исключить. Это ограничивает репрезентативность выборки, что учитывается в выводах. Журнал решения и SHA256 изображения сохраняются; ранее проверенная страница не требует повторной оценки. Ни один ответ модели не используется для отбора.

In [ ]:
from datasets import load_dataset
import io
if DEVICE.type != "cuda":
    raise RuntimeError("Полный эксперимент требует T4. Метрики ANLS и resize можно проверить на CPU отдельно.")
dataset_id = "lmms-lab-encoder/DocVQA"
docs = load_dataset(dataset_id,"DocVQA",split="validation",revision=REVISIONS[dataset_id])
review_file = OUT/"privacy_review.json"
privacy = json.loads(review_file.read_text(encoding="utf-8")) if review_file.exists() else {}
selection = []
order = np.random.default_rng(SEED).permutation(len(docs)).tolist()
for index in order:
    row = docs[index]
    image = row["image"].convert("RGB")
    buffer = io.BytesIO()
    image.save(buffer,format="PNG")
    digest = hashlib.sha256(buffer.getvalue()).hexdigest()
    key = str(row["docId"])+":"+digest
    if key not in privacy:
        display(image)
        print(row["question"],row["answers"])
        answer = input("Вся страница не содержит персональных данных? yes/no: ").strip().lower()
        while answer not in {"yes","no"}:
            answer = input("Введите yes или no после просмотра полной страницы: ").strip().lower()
        privacy[key] = {"approved":answer=="yes","sha256":digest,"doc_id":str(row["docId"])}
        save_json("privacy_review.json",privacy)
    if privacy[key]["approved"]:
        selection.append({"index":index,"question_id":str(row["questionId"]),"question":row["question"],
                          "references":row["answers"],"numeric":numeric_answer(row["answers"]),"image_sha256":digest})
    if len(selection)==100:
        break
assert len(selection)==100 and len({r["question_id"] for r in selection})==100
save_json("sample.json",{"dataset":dataset_id,"revision":REVISIONS[dataset_id],"seed":SEED,"questions":selection})

In [ ]:
model, processor = load_vlm()
answers_file = OUT/"answers.json"
answers = json.loads(answers_file.read_text(encoding="utf-8")) if answers_file.exists() else []
done = {(r["resolution"],r["question_id"]) for r in answers if r["status"]=="ok"}
for resolution,cap in PIXELS.items():
    processor.image_processor.max_pixels = cap
    warmup = docs[selection[0]["index"]]["image"].convert("RGB")
    try:
        vlm_answer(model,processor,warmup,"Read this document. Answer with one word.",8)
    except torch.OutOfMemoryError:
        torch.cuda.empty_cache()
    for record in tqdm(selection,desc=resolution):
        if (resolution,record["question_id"]) in done:
            continue
        image = docs[record["index"]]["image"].convert("RGB")
        prompt = record["question"]+" Answer with a single word or phrase. Do not explain."
        row = {**record,"resolution":resolution,"max_pixels":cap}
        try:
            result = vlm_answer(model,processor,image,prompt)
            assert result["processed_height"]*result["processed_width"] <= cap
            row.update(result,status="ok",ANLS=anls(result["answer"],record["references"]),EM=exact_match(result["answer"],record["references"]))
        except torch.OutOfMemoryError:
            torch.cuda.empty_cache()
            row.update(status="out_of_memory")
        answers = [r for r in answers if (r["resolution"],r["question_id"])!=(resolution,record["question_id"])]
        answers.append(row)
        save_json("answers.json",answers)

## 6. Полученные результаты
Время — синхронизированная генерация после переноса входов на GPU, включая prefill, без загрузки изображения и процессора. Измеряется фактическое N′ = t·h·w/merge_size². max_pixels является верхней границей, а не требованием увеличивать каждую страницу до этого размера. OOM сохраняется как незавершённый запуск, а не как неверный ответ; полные агрегаты не выводятся при пропущенных вопросах.

In [ ]:
frame = pd.DataFrame(answers)
display(frame.groupby(["resolution","status"]).size().rename("n").reset_index())
assert len(frame)==300 and frame.status.eq("ok").all(), "Есть незавершённые ответы. Полный эксперимент не засчитывается."
summary = frame.groupby("resolution",sort=False).agg(ANLS=("ANLS","mean"),EM=("EM","mean"),
                    visual_tokens=("visual_tokens","mean"),seconds=("seconds","mean"),n=("question_id","size")).reset_index()
summary.to_csv(OUT/"metrics.csv",index=False)
numeric = frame[frame.numeric].groupby("resolution",sort=False).agg(ANLS=("ANLS","mean"),EM=("EM","mean"),n=("question_id","size")).reset_index()
numeric.to_csv(OUT/"numeric_metrics.csv",index=False)
display(summary)
display(numeric)
for metric,label,name in [("ANLS","ANLS","anls_tokens.png"),("seconds","Generation time, s","time_tokens.png")]:
    plt.figure(figsize=(6,4))
    plt.plot(summary.visual_tokens,summary[metric],"o-")
    for row in summary.itertuples():
        plt.annotate(row.resolution,(row.visual_tokens,getattr(row,metric)))
    plt.xlabel("Mean visual tokens N′")
    plt.ylabel(label)
    save_figure(name)
paired = frame.pivot(index="question_id",columns="resolution",values="ANLS")
rng = np.random.default_rng(SEED)
gain = (paired["4 MP"]-paired["1 MP"]).to_numpy()
bootstrap = gain[rng.integers(0,len(gain),size=(2000,len(gain)))].mean(axis=1)
interval = np.quantile(bootstrap,[0.025,0.975]).tolist()
conclusion = (f"Прирост ANLS от 1 к 4 MP: {gain.mean():.4f}; парный bootstrap 95% интервал {interval}. "
              "Три ограничения дают только грубую оценку насыщения; не доказывают универсальное положение колена. "
              "Если N′ не изменилось, сравнение отражает одинаковое фактическое разрешение.")
middle_gain = float((paired["1 MP"]-paired["0.25 MP"]).mean())
if middle_gain>0.01 and interval[0]>-0.01 and interval[1]<0.01:
    conclusion += " В пределах этих трёх режимов наблюдаемое насыщение начинается около 1 MP."
else:
    conclusion += " Убедительное колено около 1 MP по этому критерию не подтверждено."
save_json("analysis.json",{"conclusion":conclusion,"gain_1_to_4":float(gain.mean()),"bootstrap95":interval})
display(Markdown(conclusion))

## 7. Анализ и выводы
Для числовых полей мягкое совпадение может скрывать существенную ошибку значения: рядом нужно читать EM. Следующий разбор выбирает до 10 ошибок минимального разрешения по EM. Причины фиксируются по изображению, а не выводятся автоматически из ANLS.

Числовая подвыборка определяется заранее: все эталонные ответы содержат цифры и только допустимую пунктуацию, валюты или пробелы. Ответы с единицами, записанными словами, сюда не включены. Насыщение около лимита 1 MP допускается, только если средний прирост 0.25→1 MP больше 0.01, а весь парный 95% интервал для прироста 1→4 MP лежит внутри (−0.01, 0.01). Лимит max_pixels является верхней границей: одинаковое фактическое число токенов не доказывает насыщение качества.

In [ ]:
low = frame[(frame.resolution=="0.25 MP") & (frame.EM==0)].sort_values(["ANLS","question_id"]).head(10)
high = frame[frame.resolution=="4 MP"].set_index("question_id")
review_path = OUT/"error_review.json"
reviews = json.loads(review_path.read_text(encoding="utf-8")) if review_path.exists() else []
for row in low.to_dict("records"):
    top = high.loc[row["question_id"]]
    display(docs[row["index"]]["image"])
    print(row["question"],"\nЭталоны:",row["references"],"\n0.25 MP:",row["answer"],"\n4 MP:",top.answer)
    if row["question_id"] not in {r["question_id"] for r in reviews}:
        explanation = input("Причина по изображению (мелкий текст, чтение, рассуждение, неясно) и обоснование: ").strip()
        while not explanation:
            explanation = input("Введите обоснование или 'причина не установлена': ").strip()
        reviews.append({"question_id":row["question_id"],"low_answer":row["answer"],"high_answer":top.answer,
                        "references":row["references"],"fixed_EM":bool(top.EM),"ANLS_gain":float(top.ANLS-row["ANLS"]),"reason":explanation})
        save_json("error_review.json",reviews)
display(pd.DataFrame(reviews))
if len(low)<10:
    print(f"Ошибок при минимальном разрешении всего {len(low)}; дополнительные ошибки не придуманы.")

## 8. Контрольные вопросы
**Почему ANLS выше EM и где это опасно?** ANLS даёт частичный балл за похожие строки. Это опасно для сумм, дат, идентификаторов и дозировок: одна цифра может полностью менять смысл. При совершенно точных ответах метрики равны.

**Цена удвоения стороны?** Число патчей обычно растёт примерно в четыре раза, плотная матрица attention — в шестнадцать. Реальное время Qwen2.5-VL зависит также от оконного attention визуального энкодера, длины ответа и оборудования.

In [6]:
findings = ["Выполнены CPU-проверки: ANLS('1234','1235')=0.75, EM=0; проверены порог 0.5, пустые строки, несколько эталонов и девять случаев resize с кратностью 28."]
if (OUT/"metrics.csv").exists():
    table = pd.read_csv(OUT/"metrics.csv").drop(columns="n")
    findings.append(json.loads((OUT/"analysis.json").read_text(encoding="utf-8"))["conclusion"])
    numeric = pd.read_csv(OUT/"numeric_metrics.csv")
    findings.append("Числовые поля: "+numeric.to_string(index=False))
else:
    findings.append("Таблица ниже — проверка процессора на пустом синтетическом изображении 1500×2000. Это не результаты DocVQA и не ответы VLM.")
    table = pd.DataFrame(json.loads((OUT/"processor_tests.json").read_text())) if (OUT/"processor_tests.json").exists() else pd.DataFrame([{"test":"ANLS 1234 vs 1235","value":0.75},{"test":"EM 1234 vs 1235","value":0.0}])
    findings += ["Инференс на 100 вопросах при 0.25, 1 и 4 MP не выполнен: в текущей среде нет CUDA GPU. Выборка без персональных данных также ещё не подтверждена визуальной проверкой.",
                 "Нет фактических ANLS, EM, времени модели, графиков и десяти разобранных ошибок. Код сохраняет вопросы, эталоны, ответы, N′, время, ошибки памяти и отдельные числовые метрики.",
                 "Вывод о колене кривой пока сделать нельзя. Данные трёх разрешений и парный bootstrap предусмотрены в notebook."]
write_report("ПЗ-4. Разрешение документов", "Изучить зависимость качества от числа визуальных токенов.",
             "Реализованы ANLS, протокол трёх разрешений, проверка документов и анализ числовых ответов.",findings,table,OUT/"anls_tokens.png")

PDF: C:\Users\darri\PycharmProjects\cv061126\PZ_4\report.pdf
